In [ ]:
# -*- coding: utf-8 -*-
"""
光场强度交叉矩阵分析 - 支持多波长 (波长标记显示版本) - 完整修改版
基于光场仿真数据 - 增强一致性验证和优化
"""
import os
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import pandas as pd

import torch
import numpy as np
import os
import time
import json

from model import MultiModeMultiWavelengthModel
from config import Config
from mask_loader import MaskLoader
from label_utils import visualize_labels
from simulator import Simulator

start_time = time.time()

# 设置随机种子，确保结果可重现
torch.manual_seed(42)
np.random.seed(42)

print("=" * 60)
print("多模式多波长光场调制系统 - 训练-仿真集成版 (完整修改版 - 增强一致性)")
print("=" * 60)

# ===== 创建增强配置 =====

# 基本参数
num_modes = 3                                    # 模式数量
wavelengths = np.array([1310e-9, 1550e-9])      # 波长列表(m) - 恢复两个波长
base_wavelength_idx = 1                          # 基准波长索引

# 空间参数
field_size = 50                                  # 场大小(像素)
layer_size = 300                                 # 层大小(像素)
focus_radius = 10                                # 焦点半径(像素)
detectsize = 15                                  # 检测区域大小(像素)

# 物理参数
z_layers = 40e-6                                 # 层间距离(m)
z_prop = 150e-6                                  # 传播距离(m)
z_step = 20e-6                                   # 传播步长(m)
pixel_size = 1e-6                                # 像素大小(m)

# 检测区域偏移 - 修改：为每个波长提供精确偏移
offsets = [(0, 0), (0, 0)]                       # 每个波长的检测区域偏移

# 训练参数
learning_rate = 0.01                             # 学习率
lr_decay = 0.99                                  # 学习率衰减
epochs = 700                                     # 训练轮数
batch_size = 16                                  # 批量大小

# Zero Padding 参数
padding_ratio = 0.01                             # Padding 比例 (1%)
use_apodization = True                           # 启用边界衰减
apodization_width = 15                           # 衰减宽度

# MaskLoader 参数
fallback_focal_lengths = [40e-6, 60e-6, 80e-6, 100e-6, 120e-6]  # 备用掩码的焦距列表
default_num_layers = 3                           # 默认层数

# 保存参数
save_dir = f"./results/{len(wavelengths)}_wl_basewl_{wavelengths[base_wavelength_idx]}_z_prop_{z_prop}_focus_{focus_radius}/"
flag_savemat = True

# ===== 创建Config对象 =====
config = Config(
    num_modes=num_modes,
    wavelengths=wavelengths,
    base_wavelength_idx=base_wavelength_idx,
    field_size=field_size,
    layer_size=layer_size,
    focus_radius=focus_radius,
    detectsize=detectsize,
    z_layers=z_layers,
    z_prop=z_prop,
    z_step=z_step,
    pixel_size=pixel_size,
    offsets=offsets,
    learning_rate=learning_rate,
    lr_decay=lr_decay,
    epochs=epochs,
    batch_size=batch_size,
    padding_ratio=padding_ratio,
    use_apodization=use_apodization,
    apodization_width=apodization_width,
    fallback_focal_lengths=fallback_focal_lengths,
    default_num_layers=default_num_layers,
    save_dir=save_dir,
    flag_savemat=flag_savemat
)

print(f"✅ 配置创建成功！")
print(f"波长数量: {len(config.wavelengths)}")
print(f"模式数量: {config.num_modes}")
print(f"保存目录: {config.save_dir}")

# ===== 新增：增强的检测区域创建函数 =====
def create_enhanced_evaluation_regions_multiwavelength(field_size, layer_size, focus_radius, detectsize, 
                                                      wavelengths, num_modes, offsets=None, 
                                                      separation_factor=1.5, verify_consistency=True):
    """
    创建增强的多波长检测区域 - 确保完全一致性
    
    参数:
        field_size: 输入场大小
        layer_size: 层大小
        focus_radius: 焦点半径
        detectsize: 检测区域大小
        wavelengths: 波长列表
        num_modes: 模式数量
        offsets: 每个波长的偏移
        separation_factor: 分离因子，确保区域不重叠
        verify_consistency: 是否验证一致性
    
    返回:
        evaluation_regions: 检测区域列表
        detector_labels: 检测器标签映射
        region_info: 详细区域信息
    """
    
    print(f"\n🔧 创建增强的多波长检测区域...")
    
    if offsets is None:
        offsets = [(0, 0)] * len(wavelengths)
    
    num_wavelengths = len(wavelengths)
    wavelengths_nm = [int(wl * 1e9) for wl in wavelengths]
    
    # 计算安全的区域分布
    # 波长维度：沿X轴分布
    wl_spacing = layer_size // (num_wavelengths + 1)  # 增加边距
    # 模式维度：沿Y轴分布  
    mode_spacing = layer_size // (num_modes + 1)      # 增加边距
    
    print(f"  区域分布参数:")
    print(f"    波长间距: {wl_spacing} pixels")
    print(f"    模式间距: {mode_spacing} pixels")
    print(f"    检测区域大小: {detectsize}×{detectsize}")
    print(f"    分离因子: {separation_factor}")
    
    evaluation_regions = []
    detector_labels = {}
    region_info = []
    
    detector_idx = 0
    
    for wl_idx, (wavelength, wavelength_nm) in enumerate(zip(wavelengths, wavelengths_nm)):
        for mode_idx in range(num_modes):
            # 计算基础中心位置
            base_center_x = (wl_idx + 1) * wl_spacing
            base_center_y = (mode_idx + 1) * mode_spacing
            
            # 应用偏移
            offset_x, offset_y = offsets[wl_idx] if wl_idx < len(offsets) else (0, 0)
            center_x = base_center_x + offset_x
            center_y = base_center_y + offset_y
            
            # 计算检测区域边界（确保在有效范围内）
            half_detect = detectsize // 2
            x_start = max(0, center_x - half_detect)
            x_end = min(layer_size, center_x + half_detect)
            y_start = max(0, center_y - half_detect)
            y_end = min(layer_size, center_y + half_detect)
            
            # 确保区域大小一致
            actual_width = x_end - x_start
            actual_height = y_end - y_start
            
            if actual_width < detectsize or actual_height < detectsize:
                print(f"  ⚠️ 警告: 检测器{detector_idx}区域被裁剪 ({actual_width}×{actual_height})")
            
            # 添加到列表
            region = (int(x_start), int(x_end), int(y_start), int(y_end))
            evaluation_regions.append(region)
            
            # 创建标签
            detector_labels[detector_idx] = f'{wavelength_nm}nm-Det{mode_idx+1}'
            
            # 记录详细信息
            region_info.append({
                'detector_idx': detector_idx,
                'wavelength_nm': wavelength_nm,
                'wavelength_idx': wl_idx,
                'mode_idx': mode_idx,
                'center': (center_x, center_y),
                'region': region,
                'size': (actual_width, actual_height),
                'label': detector_labels[detector_idx]
            })
            
            print(f"    检测器{detector_idx} ({detector_labels[detector_idx]}):")
            print(f"      中心: ({center_x:.1f}, {center_y:.1f})")
            print(f"      区域: [{x_start}-{x_end}, {y_start}-{y_end}]")
            print(f"      尺寸: {actual_width}×{actual_height}")
            
            detector_idx += 1
    
    # 验证一致性
    if verify_consistency:
        print(f"\n🔍 验证检测区域一致性...")
        consistency_issues = verify_region_consistency(evaluation_regions, region_info)
        
        if consistency_issues:
            print(f"  ❌ 发现 {len(consistency_issues)} 个一致性问题:")
            for issue in consistency_issues:
                print(f"    - {issue}")
        else:
            print(f"  ✅ 所有检测区域一致性验证通过")
    
    print(f"  ✅ 创建完成: {len(evaluation_regions)} 个检测区域")
    
    return evaluation_regions, detector_labels, region_info

def verify_region_consistency(evaluation_regions, region_info):
    """验证检测区域一致性"""
    
    issues = []
    
    # 1. 检查区域重叠
    for i, region1 in enumerate(evaluation_regions):
        for j, region2 in enumerate(evaluation_regions[i+1:], i+1):
            x1_start, x1_end, y1_start, y1_end = region1
            x2_start, x2_end, y2_start, y2_end = region2
            
            # 计算重叠
            overlap_x = max(0, min(x1_end, x2_end) - max(x1_start, x2_start))
            overlap_y = max(0, min(y1_end, y2_end) - max(y1_start, y2_start))
            
            if overlap_x > 0 and overlap_y > 0:
                overlap_area = overlap_x * overlap_y
                area1 = (x1_end - x1_start) * (y1_end - y1_start)
                overlap_ratio = overlap_area / area1
                
                if overlap_ratio > 0.05:  # 5% 重叠阈值
                    issues.append(f"区域重叠: 检测器{i}与检测器{j} (重叠率: {overlap_ratio:.2%})")
    
    # 2. 检查区域大小一致性
    sizes = [(info['size'][0], info['size'][1]) for info in region_info]
    unique_sizes = set(sizes)
    
    if len(unique_sizes) > 1:
        issues.append(f"区域大小不一致: 发现 {len(unique_sizes)} 种不同大小 {unique_sizes}")
    
    # 3. 检查边界问题
    for i, info in enumerate(region_info):
        x_start, x_end, y_start, y_end = info['region']
        if x_start <= 0 or y_start <= 0 or x_end >= 300 or y_end >= 300:
            issues.append(f"边界问题: 检测器{i} 触及图像边界")
    
    return issues

def create_consistent_labels_multiwavelength(layer_size, focus_radius, evaluation_regions, region_info, num_modes):
    """
    创建与检测区域完全一致的标签
    
    参数:
        layer_size: 层大小
        focus_radius: 焦点半径
        evaluation_regions: 检测区域列表
        region_info: 区域详细信息
        num_modes: 模式数量
    
    返回:
        labels: 标签张量 [模式, 波长, Y, X]
        label_info: 标签详细信息
    """
    
    print(f"\n🏷️ 创建一致性标签...")
    
    # 确定波长数量
    wavelengths_in_regions = set([info['wavelength_nm'] for info in region_info])
    num_wavelengths = len(wavelengths_in_regions)
    sorted_wavelengths = sorted(wavelengths_in_regions)
    
    # 创建标签张量
    labels = torch.zeros(num_modes, num_wavelengths, layer_size, layer_size)
    
    # 创建坐标网格
    y, x = torch.meshgrid(
        torch.arange(layer_size, dtype=torch.float32),
        torch.arange(layer_size, dtype=torch.float32),
        indexing='ij'
    )
    
    label_info = []
    
    # 为每个检测区域创建对应的标签
    for info in region_info:
        wavelength_nm = info['wavelength_nm']
        mode_idx = info['mode_idx']
        center_x, center_y = info['center']
        
        # 找到波长索引
        wl_idx = sorted_wavelengths.index(wavelength_nm)
        
        # 生成高斯标签（中心对齐到检测区域中心）
        distance = torch.sqrt((x - center_x)**2 + (y - center_y)**2)
        sigma = focus_radius / 3.0  # 3-sigma规则
        gaussian = torch.exp(-distance**2 / (2 * sigma**2))
        
        # 归一化
        gaussian = gaussian / gaussian.max()
        
        # 分配到对应的标签张量位置
        labels[mode_idx, wl_idx] = gaussian
        
        # 验证标签峰值位置
        peak_y, peak_x = torch.where(gaussian == gaussian.max())
        peak_center_x = peak_x[0].item()
        peak_center_y = peak_y[0].item()
        
        # 计算标签在检测区域内的覆盖情况
        x_start, x_end, y_start, y_end = info['region']
        region_label = gaussian[y_start:y_end, x_start:x_end]
        coverage_ratio = torch.sum(region_label > 0.1) / region_label.numel()
        intensity_ratio = torch.sum(region_label) / torch.sum(gaussian)
        
        label_info.append({
            'detector_idx': info['detector_idx'],
            'label': info['label'],
            'mode_idx': mode_idx,
            'wl_idx': wl_idx,
            'wavelength_nm': wavelength_nm,
            'region_center': (center_x, center_y),
            'label_peak': (peak_center_x, peak_center_y),
            'center_deviation': np.sqrt((peak_center_x - center_x)**2 + (peak_center_y - center_y)**2),
            'coverage_ratio': coverage_ratio.item(),
            'intensity_ratio': intensity_ratio.item()
        })
        
        print(f"  标签{info['detector_idx']} ({info['label']}):")
        print(f"    区域中心: ({center_x:.1f}, {center_y:.1f})")
        print(f"    标签峰值: ({peak_center_x:.1f}, {peak_center_y:.1f})")
        print(f"    中心偏差: {label_info[-1]['center_deviation']:.2f} pixels")
        print(f"    覆盖率: {coverage_ratio:.3f}")
        print(f"    强度占比: {intensity_ratio:.3f}")
    
    print(f"  ✅ 标签创建完成: {labels.shape}")
    
    return labels, label_info

# ===== 修改后的数据加载函数 =====
def load_field_data_multiwavelength_enhanced(config):
    """加载多波长光场数据 - 增强版"""
    
    print("🔍 加载多波长光场数据 (增强版)...")
    
    # 查找所有.npy文件
    field_files = []
    base_dir = config.save_dir
    
    if not os.path.exists(base_dir):
        print(f"❌ 目录不存在: {base_dir}")
        return {}
    
    for file in os.listdir(base_dir):
        if file.endswith('.npy'):
            field_files.append(os.path.join(base_dir, file))
    
    print(f"✅ 找到 {len(field_files)} 个光场数据文件")
    
    if len(field_files) == 0:
        print("❌ 未找到任何.npy文件")
        return {}
    
    # 按波长、层数和模式组织数据: wavelength -> layer -> mode -> data_list
    field_data = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))
    
    successful_loads = 0
    failed_loads = 0
    
    for file_path in field_files:
        filename = os.path.basename(file_path)
        
        try:
            # 解析文件名获取波长、层数和模式信息
            parts = filename.replace('.npy', '').split('_')
            
            wavelength_nm = None
            mode_num = None
            layer_num = None
            
            for part in parts:
                if part.endswith('nm'):
                    wavelength_nm = int(part.replace('nm', ''))
                elif part.startswith('mode'):
                    mode_num = int(part.replace('mode', ''))
                elif part.endswith('layers'):
                    layer_num = int(part.replace('layers', ''))
            
            if wavelength_nm is None or mode_num is None or layer_num is None:
                print(f"❌ 跳过文件 (解析失败): {filename}")
                failed_loads += 1
                continue
            
            # 加载光场数据
            field = np.load(file_path)
            
            if field.size == 0:
                print(f"❌ 跳过空文件: {filename}")
                failed_loads += 1
                continue
            
            # 计算强度
            if np.iscomplexobj(field):
                intensity = np.abs(field)**2
            else:
                intensity = field**2
            
            # 数据质量检查
            if np.isnan(intensity).any() or np.isinf(intensity).any():
                print(f"❌ 跳过无效数据文件: {filename}")
                failed_loads += 1
                continue
            
            field_data[wavelength_nm][layer_num][mode_num].append({
                'intensity': intensity,
                'field': field,
                'filename': filename,
                'wavelength': wavelength_nm,
                'layer': layer_num,
                'mode': mode_num,
                'max_intensity': np.max(intensity),
                'total_power': np.sum(intensity),
                'mean_intensity': np.mean(intensity),
                'shape': intensity.shape
            })
            
            successful_loads += 1
            print(f"  ✓ {wavelength_nm}nm {layer_num}层 模式{mode_num}: {field.shape}, 最大强度: {np.max(intensity):.6f}")
            
        except Exception as e:
            print(f"❌ 处理失败 {filename}: {e}")
            failed_loads += 1
            continue
    
    print(f"\n📊 数据加载统计:")
    print(f"  成功加载: {successful_loads} 个文件")
    print(f"  加载失败: {failed_loads} 个文件")
    print(f"  波长数量: {len(field_data)}")
    print(f"  总配置数: {sum(len(wl_data) for wl_data in field_data.values())}")
    
    return field_data

# ===== 修改后的主要分析函数 =====
def calculate_intensity_cross_matrix_multiwavelength_enhanced(field_data, config):
    """多波长交叉矩阵计算 - 增强版，确保完全一致性"""
    
    print("\n📊 计算多波长强度交叉矩阵 (增强版)...")
    
    # 获取参数
    num_modes = config.num_modes
    focus_radius = config.focus_radius
    detectsize = config.detectsize
    layer_size = config.layer_size
    wavelengths = config.wavelengths
    offsets = config.offsets
    
    num_wavelengths = len(wavelengths)
    wavelengths_nm = [int(wl * 1e9) for wl in wavelengths]
    
    # 获取所有波长、层数和模式
    all_wavelengths = sorted(field_data.keys())
    all_layers = sorted(set().union(*[wl_data.keys() for wl_data in field_data.values()]))
    all_modes = sorted(set().union(*[
        set().union(*[layer_data.keys() for layer_data in wl_data.values()])
        for wl_data in field_data.values()
    ]))
    
    print(f"📋 分析计划:")
    print(f"  波长: {all_wavelengths}nm")
    print(f"  层数: {all_layers}")
    print(f"  模式: {all_modes}")
    print(f"  总检测器数: {num_wavelengths * num_modes}")
    
    # 使用增强的检测区域创建函数
    evaluation_regions, detector_labels, region_info = create_enhanced_evaluation_regions_multiwavelength(
        config.field_size, layer_size, focus_radius, detectsize, 
        wavelengths, num_modes, offsets, verify_consistency=True
    )
    
    # 创建一致的标签
    consistent_labels, label_info = create_consistent_labels_multiwavelength(
        layer_size, focus_radius, evaluation_regions, region_info, num_modes
    )
    
    # 结果存储
    cross_matrices = {}
    normalized_matrices = {}
    visibility_results = {}
    consistency_reports = {}
    
    # 处理每个层配置
    for layer_idx, layer_num in enumerate(all_layers):
        print(f"\n🔄 [{layer_idx+1}/{len(all_layers)}] 处理 {layer_num} 层配置...")
        
        try:
            # 创建交叉矩阵
            total_detectors = len(evaluation_regions)
            intensity_matrix = np.zeros((total_detectors, num_modes))
            
            # 一致性报告
            consistency_report = {
                'layer': layer_num,
                'processed_combinations': 0,
                'failed_combinations': 0,
                'region_consistency_scores': [],
                'label_consistency_scores': []
            }
            
            # 创建保存目录
            vis_save_dir = os.path.join(config.save_dir, f"enhanced_detection_visualization_layer_{layer_num}")
            
            # 处理每个波长的数据
            for wavelength in all_wavelengths:
                if wavelength not in field_data or layer_num not in field_data[wavelength]:
                    continue
                
                layer_data = field_data[wavelength][layer_num]
                
                # 生成增强的可视化图片
                visualize_enhanced_detection_regions(
                    field_data, wavelength, layer_num, evaluation_regions, 
                    detector_labels, region_info, label_info, vis_save_dir
                )
                
                # 计算每个输入模式的强度分布
                for input_mode_idx, input_mode in enumerate(all_modes):
                    if input_mode not in layer_data:
                        continue
                        
                    mode_data_list = layer_data[input_mode]
                    if not mode_data_list:
                        continue
                    
                    try:
                        # 计算区域强度
                        total_intensities = []
                        for data_entry in mode_data_list:
                            intensity = data_entry['intensity']
                            region_intensities = evaluate_output_enhanced(intensity, evaluation_regions)
                            total_intensities.append(region_intensities)
                        
                        if total_intensities:
                            avg_intensities = np.mean(total_intensities, axis=0)
                            
                            # 填充交叉矩阵
                            for detector_idx in range(len(avg_intensities)):
                                if input_mode_idx < num_modes and detector_idx < total_detectors:
                                    intensity_matrix[detector_idx, input_mode_idx] = avg_intensities[detector_idx]
                            
                            consistency_report['processed_combinations'] += 1
                            print(f"    ✓ {wavelength}nm 模式{input_mode}: 平均强度 {np.mean(avg_intensities):.6f}")
                        
                    except Exception as e:
                        print(f"    ❌ {wavelength}nm 模式{input_mode} 处理失败: {e}")
                        consistency_report['failed_combinations'] += 1
                        continue
            
            print(f"  📊 处理统计: 成功 {consistency_report['processed_combinations']}, 失败 {consistency_report['failed_combinations']}")
            
            # 保存结果
            cross_matrices[layer_num] = intensity_matrix.copy()
            
            # 归一化：按列归一化
            normalized_matrix = np.zeros_like(intensity_matrix)
            for col in range(intensity_matrix.shape[1]):
                col_sum = np.sum(intensity_matrix[:, col])
                if col_sum > 0:
                    normalized_matrix[:, col] = intensity_matrix[:, col] / col_sum
            
            normalized_matrices[layer_num] = normalized_matrix
            
            # 计算增强的可见度
            visibility = calculate_enhanced_visibility(normalized_matrix, region_info, num_modes)
            visibility_results[layer_num] = visibility
            
            # 保存一致性报告
            consistency_reports[layer_num] = consistency_report
            
            print(f"  🎯 增强可见度: {visibility:.4f}")
            
        except Exception as e:
            print(f"  ❌ 层{layer_num}处理失败: {e}")
            continue
    
    print(f"\n✅ 增强版多波长交叉矩阵计算完成！处理了 {len(cross_matrices)} 个配置")
    
    return (cross_matrices, normalized_matrices, visibility_results, 
            all_wavelengths, all_layers, all_modes, detector_labels, 
            evaluation_regions, region_info, label_info, consistency_reports)

def evaluate_output_enhanced(intensity_field, evaluation_regions):
    """增强的输出评估函数"""
    
    region_intensities = []
    
    for region in evaluation_regions:
        x_start, x_end, y_start, y_end = region
        
        # 确保索引在有效范围内
        x_start = max(0, min(x_start, intensity_field.shape[1]-1))
        x_end = max(0, min(x_end, intensity_field.shape[1]))
        y_start = max(0, min(y_start, intensity_field.shape[0]-1))
        y_end = max(0, min(y_end, intensity_field.shape[0]))
        
        # 提取区域并计算增强的平均强度
        if x_end > x_start and y_end > y_start:
            region_data = intensity_field[y_start:y_end, x_start:x_end]
            
            # 使用加权平均（中心权重更高）
            center_y = (y_end + y_start) // 2 - y_start
            center_x = (x_end + x_start) // 2 - x_start
            
            y_coords, x_coords = np.meshgrid(
                np.arange(region_data.shape[0]), 
                np.arange(region_data.shape[1]), 
                indexing='ij'
            )
            
            # 距离权重
            distances = np.sqrt((x_coords - center_x)**2 + (y_coords - center_y)**2)
            max_distance = np.max(distances)
            weights = 1.0 - (distances / max_distance) if max_distance > 0 else np.ones_like(distances)
            
            # 加权平均强度
            weighted_intensity = np.sum(region_data * weights) / np.sum(weights)
            region_intensities.append(weighted_intensity)
        else:
            region_intensities.append(0.0)
    
    return np.array(region_intensities)

def calculate_enhanced_visibility(normalized_matrix, region_info, num_modes):
    """计算增强的可见度指标"""
    
    # 按波长分组计算可见度
    wavelength_groups = {}
    for info in region_info:
        wl = info['wavelength_nm']
        if wl not in wavelength_groups:
            wavelength_groups[wl] = []
        wavelength_groups[wl].append(info)
    
    visibility_values = []
    
    for wavelength_nm, wl_regions in wavelength_groups.items():
        # 对于每个波长，计算其内部的对角线可见度
        for info in wl_regions:
            detector_idx = info['detector_idx']
            mode_idx = info['mode_idx']
            
            if detector_idx < normalized_matrix.shape[0] and mode_idx < normalized_matrix.shape[1]:
                diagonal_value = normalized_matrix[detector_idx, mode_idx]
                
                # 计算该检测器对其他模式的串扰
                crosstalk_values = []
                for other_mode_idx in range(num_modes):
                    if other_mode_idx != mode_idx:
                        crosstalk_values.append(normalized_matrix[detector_idx, other_mode_idx])
                
                # 可见度 = 对角线值 / (对角线值 + 最大串扰值)
                max_crosstalk = max(crosstalk_values) if crosstalk_values else 0
                visibility = diagonal_value / (diagonal_value + max_crosstalk) if (diagonal_value + max_crosstalk) > 0 else 0
                
                visibility_values.append(visibility)
    
    return np.mean(visibility_values) if visibility_values else 0

def visualize_enhanced_detection_regions(field_data, wavelength, layer_num, evaluation_regions, 
                                       detector_labels, region_info, label_info, save_dir):
    """增强的检测区域可视化"""
    
    print(f"   🎨 生成{wavelength}nm第{layer_num}层的增强可视化...")
    
    if wavelength not in field_data or layer_num not in field_data[wavelength]:
        return
    
    layer_data = field_data[wavelength][layer_num]
    os.makedirs(save_dir, exist_ok=True)
    
    # 1. 生成一致性验证图
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle(f'{wavelength}nm Layer {layer_num} - Enhanced Consistency Verification', fontsize=16)
    
    # 合成所有模式的强度
    combined_intensity = None
    valid_modes = 0
    
    for mode_idx in range(1, 4):
        if mode_idx in layer_data and layer_data[mode_idx]:
            mode_intensity = layer_data[mode_idx][0]['intensity']
            
            if combined_intensity is None:
                combined_intensity = np.zeros_like(mode_intensity)
            
            combined_intensity += mode_intensity
            valid_modes += 1
    
    if combined_intensity is None:
        return
    
    # 绘制各个子图
    plots_info = [
        ('Original Intensity', combined_intensity, 'hot'),
        ('Detection Regions', combined_intensity, 'hot'),
        ('Label Overlay', combined_intensity, 'hot'),
        ('Consistency Map', combined_intensity, 'viridis'),
        ('Region Analysis', combined_intensity, 'plasma'),
        ('Quality Metrics', combined_intensity, 'coolwarm')
    ]
    
    for idx, (title, data, cmap) in enumerate(plots_info):
        row, col = idx // 3, idx % 3
        ax = axes[row, col]
        
        im = ax.imshow(data, cmap=cmap, origin='upper')
        ax.set_title(title)
        ax.axis('off')
        
        if idx == 1:  # Detection Regions
            colors = ['cyan', 'lime', 'yellow', 'magenta', 'orange', 'red']
            for i, (x_start, x_end, y_start, y_end) in enumerate(evaluation_regions):
                color = colors[i % len(colors)]
                rect = plt.Rectangle((x_start, y_start), x_end - x_start, y_end - y_start,
                                   linewidth=2, edgecolor=color, facecolor='none')
                ax.add_patch(rect)
                
                center_x = (x_start + x_end) / 2
                center_y = (y_start + y_end) / 2
                
                if i in detector_labels:
                    label_text = detector_labels[i]
                else:
                    label_text = f'Det{i+1}'
                    
                ax.text(center_x, center_y, label_text, 
                       ha='center', va='center', fontsize=8, 
                       color='white', weight='bold',
                       bbox=dict(boxstyle='round,pad=0.3', facecolor='black', alpha=0.7))
        
        elif idx == 2:  # Label Overlay
            # 叠加标签信息
            for info in label_info:
                if info['wavelength_nm'] == wavelength:
                    center_x, center_y = info['region_center']
                    peak_x, peak_y = info['label_peak']
                    
                    # 绘制中心点
                    ax.plot(center_x, center_y, 'ro', markersize=8, label='Region Center')
                    ax.plot(peak_x, peak_y, 'b+', markersize=10, markeredgewidth=3, label='Label Peak')
                    
                    # 绘制偏差线
                    ax.plot([center_x, peak_x], [center_y, peak_y], 'g--', linewidth=2, alpha=0.7)
                    
                    # 添加偏差标注
                    deviation = info['center_deviation']
                    ax.text(center_x + 5, center_y + 5, f'{deviation:.1f}px', 
                           fontsize=8, color='yellow', weight='bold')
        
        elif idx == 3:  # Consistency Map
            # 绘制一致性热图
            consistency_map = np.zeros_like(data)
            for info in label_info:
                if info['wavelength_nm'] == wavelength:
                    x_start, x_end, y_start, y_end = evaluation_regions[info['detector_idx']]
                    consistency_score = info['coverage_ratio'] * info['intensity_ratio']
                    consistency_map[y_start:y_end, x_start:x_end] = consistency_score
            
            im = ax.imshow(consistency_map, cmap='viridis', origin='upper', vmin=0, vmax=1)
            
        elif idx == 4:  # Region Analysis
            # 绘制区域分析
            for i, info in enumerate(region_info):
                if info['wavelength_nm'] == wavelength:
                    x_start, x_end, y_start, y_end = info['region']
                    center_x, center_y = info['center']
                    
                    # 绘制区域边界
                    rect = plt.Rectangle((x_start, y_start), x_end - x_start, y_end - y_start,
                                       linewidth=2, edgecolor='white', facecolor='none')
                    ax.add_patch(rect)
                    
                    # 添加区域信息
                    ax.text(center_x, center_y, f"Det{info['detector_idx']}\n{info['size'][0]}×{info['size'][1]}", 
                           ha='center', va='center', fontsize=7, 
                           color='white', weight='bold',
                           bbox=dict(boxstyle='round,pad=0.2', facecolor='red', alpha=0.7))
        
        elif idx == 5:  # Quality Metrics
            # 显示质量指标
            ax.text(0.1, 0.9, f'Wavelength: {wavelength}nm', transform=ax.transAxes, fontsize=12, weight='bold')
            ax.text(0.1, 0.8, f'Layer: {layer_num}', transform=ax.transAxes, fontsize=12)
            ax.text(0.1, 0.7, f'Valid Modes: {valid_modes}', transform=ax.transAxes, fontsize=12)
            
            # 显示一致性指标
            y_pos = 0.6
            for info in label_info:
                if info['wavelength_nm'] == wavelength:
                    ax.text(0.1, y_pos, f"{info['label']}: Dev={info['center_deviation']:.1f}px, Cov={info['coverage_ratio']:.2f}", 
                           transform=ax.transAxes, fontsize=10)
                    y_pos -= 0.08
        
        plt.colorbar(im, ax=ax, shrink=0.8)
    
    plt.tight_layout()
    
    # 保存增强可视化图
    filename = f'{wavelength}nm_layer_{layer_num}_enhanced_consistency.png'
    save_path = os.path.join(save_dir, filename)
    plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.close()
    
    print(f"   ✅ 保存增强可视化: {filename}")

def plot_enhanced_cross_matrices(normalized_matrices, visibility_results, all_wavelengths, 
                                all_layers, all_modes, detector_labels, region_info, 
                                consistency_reports, save_dir):
    """绘制增强的交叉矩阵图表"""
    
    print("\n🎨 绘制增强的多波长强度交叉矩阵...")
    
    os.makedirs(save_dir, exist_ok=True)
    
    num_layers = len(all_layers)
    num_modes = len(all_modes)
    num_wavelengths = len(all_wavelengths)
    total_detectors = len(detector_labels)
    
    # 1. 主要交叉矩阵图
    if num_layers == 1:
        fig, ax = plt.subplots(1, 1, figsize=(12, 10))
        axes = [ax]
    else:
        fig, axes = plt.subplots(1, num_layers, figsize=(10 * num_layers, 8), constrained_layout=True)
        if num_layers == 1:
            axes = [axes]
    
    for idx, layer_num in enumerate(all_layers):
        if layer_num not in normalized_matrices:
            continue
            
        normalized_matrix = normalized_matrices[layer_num]
        ax = axes[idx]
        
        # 绘制增强热力图
        im = ax.imshow(normalized_matrix, cmap='Oranges', interpolation='nearest', 
                      vmin=0, vmax=1, origin='upper')
        
        ax.set_xlabel('Input Mode Index', fontsize=12)
        ax.set_ylabel('Detector Regions (Wavelength × Mode)', fontsize=12)
        ax.set_title(f'Layer {layer_num} - Enhanced Cross Matrix\nVisibility: {visibility_results.get(layer_num, 0):.3f}', 
                    fontsize=14)
        
        # 设置刻度标签
        ax.set_xticks(np.arange(num_modes))
        ax.set_xticklabels([f'Mode{m}' for m in all_modes])
        
        y_labels = [detector_labels.get(i, f'Det{i+1}') for i in range(total_detectors)]
        ax.set_yticks(list(range(total_detectors)))
        ax.set_yticklabels(y_labels, fontsize=10)
        
        # 添加波长分隔线
        for wl_idx in range(1, num_wavelengths):
            y_pos = wl_idx * num_modes - 0.5
            ax.axhline(y=y_pos, color='white', linewidth=3, linestyle='-')
        
        # 添加增强的数值标注
        for i in range(normalized_matrix.shape[0]):
            for j in range(normalized_matrix.shape[1]):
                value = normalized_matrix[i, j] * 100
                
                # 根据值的大小调整颜色
                text_color = 'white' if value < 50 else 'black'
                weight = 'bold' if value > 80 else 'normal'
                
                ax.text(j, i, f"{value:.1f}", ha='center', va='center', 
                       color=text_color, fontsize=9, weight=weight)
        
        # 添加对角线高亮
        for i in range(min(normalized_matrix.shape)):
            ax.add_patch(plt.Rectangle((i-0.4, i-0.4), 0.8, 0.8, 
                                     fill=False, edgecolor='red', linewidth=2))
    
    # 添加colorbar
    if num_layers > 1:
        cbar = fig.colorbar(im, ax=axes, shrink=0.6, location='right')
    else:
        cbar = plt.colorbar(im, ax=ax, shrink=0.8)
    cbar.set_label("Normalized Intensity (%)", fontsize=12)
    
    plt.suptitle(f"Enhanced Multi-wavelength Intensity Cross Matrix\n"
                f"{num_wavelengths} Wavelengths × {num_modes} Modes = {total_detectors} Detectors", 
                fontsize=16)
    
    # 保存主图
    save_path = os.path.join(save_dir, 'enhanced_intensity_cross_matrix_multiwavelength.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"  ✅ 保存: enhanced_intensity_cross_matrix_multiwavelength.png")
    
    # 2. 可见度和一致性对比图
    if len(visibility_results) > 1:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
        
        # 可见度图
        layers = list(visibility_results.keys())
        visibilities = list(visibility_results.values())
        
        ax1.plot(layers, visibilities, marker='o', linestyle='-', color='orange',
                linewidth=3, markersize=12, label=f'Enhanced Visibility')
        
        for x, y in zip(layers, visibilities):
            ax1.text(x, y + 0.02, f"{y:.3f}", ha='center', va='bottom', 
                    fontsize=12, weight='bold')
        
        ax1.set_ylim(0, 1)
        ax1.set_xlabel("Number of Layers", fontsize=12)
        ax1.set_ylabel("Enhanced Visibility", fontsize=12)
        ax1.set_title(f"Enhanced Visibility vs Layers\n({num_wavelengths} Wavelengths × {num_modes} Modes)", fontsize=14)
        ax1.grid(True, alpha=0.3)
        ax1.legend()
        
        # 一致性报告图
        success_rates = []
        for layer_num in layers:
            if layer_num in consistency_reports:
                report = consistency_reports[layer_num]
                total = report['processed_combinations'] + report['failed_combinations']
                success_rate = report['processed_combinations'] / total if total > 0 else 0
                success_rates.append(success_rate * 100)
            else:
                success_rates.append(0)
        
        ax2.bar(layers, success_rates, color='lightblue', alpha=0.7, label='Success Rate')
        
        for x, y in zip(layers, success_rates):
            ax2.text(x, y + 1, f"{y:.1f}%", ha='center', va='bottom', 
                    fontsize=11, weight='bold')
        
        ax2.set_ylim(0, 105)
        ax2.set_xlabel("Number of Layers", fontsize=12)
        ax2.set_ylabel("Processing Success Rate (%)", fontsize=12)
        ax2.set_title("Data Processing Consistency", fontsize=14)
        ax2.grid(True, alpha=0.3)
        ax2.legend()
        
        plt.tight_layout()
        
        save_path = os.path.join(save_dir, 'enhanced_visibility_consistency_analysis.png')
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
        
        print(f"  ✅ 保存: enhanced_visibility_consistency_analysis.png")
    
    # 3. 详细一致性报告图
    create_detailed_consistency_report(region_info, detector_labels, save_dir)

def create_detailed_consistency_report(region_info, detector_labels, save_dir):
    """创建详细的一致性报告图表"""
    
    print("   📊 生成详细一致性报告...")
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Detailed Consistency Analysis Report', fontsize=16)
    
    # 1. 区域大小分布
    ax1 = axes[0, 0]
    sizes = [(info['size'][0], info['size'][1]) for info in region_info]
    widths = [s[0] for s in sizes]
    heights = [s[1] for s in sizes]
    
    ax1.scatter(widths, heights, c=range(len(sizes)), cmap='viridis', s=100, alpha=0.7)
    ax1.set_xlabel('Region Width (pixels)')
    ax1.set_ylabel('Region Height (pixels)')
    ax1.set_title('Detection Region Size Distribution')
    ax1.grid(True, alpha=0.3)
    
    for i, (w, h) in enumerate(sizes):
        ax1.annotate(detector_labels.get(i, f'Det{i}'), (w, h), 
                    xytext=(5, 5), textcoords='offset points', fontsize=8)
    
    # 2. 中心位置分布
    ax2 = axes[0, 1]
    centers = [info['center'] for info in region_info]
    center_x = [c[0] for c in centers]
    center_y = [c[1] for c in centers]
    wavelengths = [info['wavelength_nm'] for info in region_info]
    
    scatter = ax2.scatter(center_x, center_y, c=wavelengths, cmap='coolwarm', s=150, alpha=0.8)
    ax2.set_xlabel('Center X (pixels)')
    ax2.set_ylabel('Center Y (pixels)')
    ax2.set_title('Detection Region Center Distribution')
    ax2.grid(True, alpha=0.3)
    
    for i, (x, y) in enumerate(centers):
        ax2.annotate(detector_labels.get(i, f'Det{i}'), (x, y), 
                    xytext=(5, 5), textcoords='offset points', fontsize=8)
    
    plt.colorbar(scatter, ax=ax2, label='Wavelength (nm)')
    
    # 3. 区域间距分析
    ax3 = axes[1, 0]
    distances = []
    labels_pairs = []
    
    for i, info1 in enumerate(region_info):
        for j, info2 in enumerate(region_info[i+1:], i+1):
            center1 = info1['center']
            center2 = info2['center']
            distance = np.sqrt((center1[0] - center2[0])**2 + (center1[1] - center2[1])**2)
            distances.append(distance)
            labels_pairs.append(f"{detector_labels.get(i, f'Det{i}')} - {detector_labels.get(j, f'Det{j}')}")
    
    ax3.hist(distances, bins=20, alpha=0.7, color='skyblue', edgecolor='black')
    ax3.set_xlabel('Distance Between Region Centers (pixels)')
    ax3.set_ylabel('Frequency')
    ax3.set_title('Inter-Region Distance Distribution')
    ax3.grid(True, alpha=0.3)
    
    # 添加统计信息
    ax3.axvline(np.mean(distances), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(distances):.1f}')
    ax3.axvline(np.median(distances), color='green', linestyle='--', linewidth=2, label=f'Median: {np.median(distances):.1f}')
    ax3.legend()
    
    # 4. 波长分组分析
    ax4 = axes[1, 1]
    wavelength_groups = {}
    for info in region_info:
        wl = info['wavelength_nm']
        if wl not in wavelength_groups:
            wavelength_groups[wl] = []
        wavelength_groups[wl].append(info)
    
    wl_names = list(wavelength_groups.keys())
    wl_counts = [len(wavelength_groups[wl]) for wl in wl_names]
    
    bars = ax4.bar(range(len(wl_names)), wl_counts, color=['orange', 'blue'], alpha=0.7)
    ax4.set_xlabel('Wavelength')
    ax4.set_ylabel('Number of Detectors')
    ax4.set_title('Detectors per Wavelength')
    ax4.set_xticks(range(len(wl_names)))
    ax4.set_xticklabels([f'{wl}nm' for wl in wl_names])
    
    for bar, count in zip(bars, wl_counts):
        height = bar.get_height()
        ax4.text(bar.get_x() + bar.get_width()/2., height + 0.05,
                f'{count}', ha='center', va='bottom', fontsize=12, weight='bold')
    
    plt.tight_layout()
    
    save_path = os.path.join(save_dir, 'detailed_consistency_report.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"   ✅ 保存: detailed_consistency_report.png")

def save_enhanced_data(cross_matrices, normalized_matrices, visibility_results, 
                      all_wavelengths, all_layers, all_modes, detector_labels, 
                      region_info, label_info, consistency_reports, save_dir):
    """保存增强版数据"""
    
    print("\n💾 保存增强版数据...")
    
    # 1. 保存numpy数据
    for layer_num in all_layers:
        if layer_num in cross_matrices:
            np.save(os.path.join(save_dir, f'enhanced_intensity_matrix_{layer_num}layers_raw.npy'), 
                   cross_matrices[layer_num])
            np.save(os.path.join(save_dir, f'enhanced_intensity_matrix_{layer_num}layers_normalized.npy'), 
                   normalized_matrices[layer_num])
    
    # 2. 保存CSV数据
    for layer_num in all_layers:
        if layer_num in normalized_matrices:
            total_detectors = len(detector_labels)
            row_labels = [detector_labels.get(i, f'Det{i+1}') for i in range(total_detectors)]
            col_labels = [f'Mode{m}' for m in all_modes]
            
            df = pd.DataFrame(
                normalized_matrices[layer_num],
                index=row_labels,
                columns=col_labels
            )
            
            csv_path = os.path.join(save_dir, f'enhanced_cross_matrix_{layer_num}layers.csv')
            df.to_csv(csv_path, encoding='utf-8-sig')
            print(f"  ✅ 保存: enhanced_cross_matrix_{layer_num}layers.csv")
    
    # 3. 保存详细报告
    detailed_report = {
        'analysis_summary': {
            'total_wavelengths': len(all_wavelengths),
            'total_layers': len(all_layers),
            'total_modes': len(all_modes),
            'total_detectors': len(detector_labels),
            'wavelengths_nm': sorted([int(wl*1e9) for wl in [1310e-9, 1550e-9]])
        },
        'visibility_results': visibility_results,
        'detector_mapping': detector_labels,
        'region_info': [
            {
                'detector_idx': info['detector_idx'],
                'label': info['label'],
                'wavelength_nm': info['wavelength_nm'],
                'mode_idx': info['mode_idx'],
                'center': info['center'],
                'region': info['region'],
                'size': info['size']
            }
            for info in region_info
        ],
        'consistency_reports': consistency_reports
    }
    
    with open(os.path.join(save_dir, 'enhanced_analysis_report.json'), 'w', encoding='utf-8') as f:
        json.dump(detailed_report, f, indent=2, ensure_ascii=False)
    
    print(f"  ✅ 保存: enhanced_analysis_report.json")
    
    # 4. 保存可见度结果
    visibility_data = []
    for layer, vis in visibility_results.items():
        success_rate = 0
        if layer in consistency_reports:
            report = consistency_reports[layer]
            total = report['processed_combinations'] + report['failed_combinations']
            success_rate = report['processed_combinations'] / total if total > 0 else 0
        
        visibility_data.append({
            'Layers': layer,
            'Enhanced_Visibility': vis,
            'Processing_Success_Rate': success_rate,
            'Total_Detectors': len(detector_labels),
            'Wavelengths': ', '.join([f'{wl}nm' for wl in sorted(all_wavelengths)])
        })
    
    if visibility_data:
        visibility_df = pd.DataFrame(visibility_data)
        visibility_path = os.path.join(save_dir, 'enhanced_visibility_results.csv')
        visibility_df.to_csv(visibility_path, index=False, encoding='utf-8-sig')
        print(f"  ✅ 保存: enhanced_visibility_results.csv")

def print_enhanced_summary(cross_matrices, normalized_matrices, visibility_results, 
                          all_wavelengths, all_modes, detector_labels, region_info, 
                          label_info, consistency_reports):
    """打印增强版摘要"""
    
    print("\n📋 增强版多波长强度交叉矩阵分析摘要")
    print("="*80)
    
    num_modes = len(all_modes)
    num_wavelengths = len(all_wavelengths)
    total_detectors = len(detector_labels)
    
    print(f"🔧 增强配置信息:")
    print(f"  波长: {sorted([int(wl*1e9) for wl in [1310e-9, 1550e-9]])}nm")
    print(f"  模式数: {num_modes}")
    print(f"  总检测器数: {total_detectors}")
    print(f"  检测器标记: {', '.join(list(detector_labels.values()))}")
    print(f"  分析配置数量: {len(cross_matrices)}")
    
    if cross_matrices:
        print(f"  层数范围: {min(cross_matrices.keys())} - {max(cross_matrices.keys())}")
    
    print(f"\n📊 增强可见度结果:")
    print("-" * 50)
    
    for layer_num in sorted(visibility_results.keys()):
        visibility = visibility_results[layer_num]
        success_rate = 0
        
        if layer_num in consistency_reports:
            report = consistency_reports[layer_num]
            total = report['processed_combinations'] + report['failed_combinations']
            success_rate = report['processed_combinations'] / total if total > 0 else 0
        
        print(f"{layer_num:2d}层: 可见度={visibility:.4f}, 成功率={success_rate:.1%}")
    
    # 找到最佳配置
    if visibility_results:
        best_layer = max(visibility_results.keys(), key=lambda k: visibility_results[k])
        best_visibility = visibility_results[best_layer]
        
        print(f"\n🏆 最佳增强配置: {best_layer}层")
        print(f"   最高可见度: {best_visibility:.4f}")
        
        if best_layer in consistency_reports:
            best_report = consistency_reports[best_layer]
            print(f"   处理成功: {best_report['processed_combinations']}")
            print(f"   处理失败: {best_report['failed_combinations']}")
    
    # 一致性统计
    print(f"\n🎯 一致性验证统计:")
    print("-" * 50)
    
    if label_info:
        center_deviations = [info['center_deviation'] for info in label_info]
        coverage_ratios = [info['coverage_ratio'] for info in label_info]
        intensity_ratios = [info['intensity_ratio'] for info in label_info]
        
        print(f"   中心偏差: 平均={np.mean(center_deviations):.2f}px, 标准差={np.std(center_deviations):.2f}px")
        print(f"   覆盖率: 平均={np.mean(coverage_ratios):.3f}, 标准差={np.std(coverage_ratios):.3f}")
        print(f"   强度比: 平均={np.mean(intensity_ratios):.3f}, 标准差={np.std(intensity_ratios):.3f}")
    
    # 区域信息统计
    print(f"\n📍 检测区域统计:")
    print("-" * 50)
    
    if region_info:
        region_sizes = [(info['size'][0] * info['size'][1]) for info in region_info]
        wavelength_distribution = {}
        
        for info in region_info:
            wl = info['wavelength_nm']
            if wl not in wavelength_distribution:
                wavelength_distribution[wl] = 0
            wavelength_distribution[wl] += 1
        
        print(f"   区域面积: 平均={np.mean(region_sizes):.0f}px², 范围={min(region_sizes)}-{max(region_sizes)}px²")
        print(f"   波长分布: {wavelength_distribution}")
    
    print("\n" + "="*80)

def main_enhanced_analysis():
    """增强版主分析函数"""
    
    print("🚀 启动增强版多波长强度交叉矩阵分析")
    print("="*80)
    
    # 配置参数
    base_dir = "/path/to/your/data"  # 请替换为实际路径
    save_dir = "/path/to/save/results"  # 请替换为实际保存路径
    
    # 分析参数
    all_wavelengths = [1310e-9, 1550e-9]  # 波长 (m)
    all_layers = [1, 2, 3, 4, 5]  # 层数
    all_modes = [1, 2, 3]  # 模式
    
    # 增强的检测区域定义 (x_start, x_end, y_start, y_end)
    enhanced_evaluation_regions = [
        (50, 150, 50, 150),    # 1310nm Mode1
        (200, 300, 50, 150),   # 1310nm Mode2  
        (350, 450, 50, 150),   # 1310nm Mode3
        (50, 150, 200, 300),   # 1550nm Mode1
        (200, 300, 200, 300),  # 1550nm Mode2
        (350, 450, 200, 300),  # 1550nm Mode3
    ]
    
    # 增强的检测器标签
    enhanced_detector_labels = {
        0: "1310nm-M1", 1: "1310nm-M2", 2: "1310nm-M3",
        3: "1550nm-M1", 4: "1550nm-M2", 5: "1550nm-M3"
    }
    
    try:
        # 1. 加载和预处理数据
        print("\n📂 加载增强版数据...")
        field_data = load_enhanced_field_data(base_dir, all_wavelengths, all_layers, all_modes)
        
        if not field_data:
            print("❌ 无法加载数据，请检查路径和文件")
            return
        
        # 2. 生成区域信息
        print("\n🎯 生成增强检测区域信息...")
        region_info = generate_enhanced_region_info(enhanced_evaluation_regions, 
                                                   enhanced_detector_labels, 
                                                   all_wavelengths, all_modes)
        
        # 3. 执行增强分析
        print("\n🔬 执行增强版强度交叉矩阵分析...")
        
        cross_matrices = {}
        normalized_matrices = {}
        visibility_results = {}
        consistency_reports = {}
        all_label_info = []
        
        for layer_num in all_layers:
            print(f"\n  分析第{layer_num}层...")
            
            # 计算交叉矩阵
            cross_matrix = calculate_enhanced_cross_matrix(
                field_data, layer_num, all_wavelengths, all_modes, 
                enhanced_evaluation_regions, enhanced_detector_labels
            )
            
            if cross_matrix is not None:
                cross_matrices[layer_num] = cross_matrix
                
                # 归一化
                normalized_matrix = normalize_enhanced_matrix(cross_matrix)
                normalized_matrices[layer_num] = normalized_matrix
                
                # 计算增强可见度
                visibility = calculate_enhanced_visibility(normalized_matrix, region_info, len(all_modes))
                visibility_results[layer_num] = visibility
                
                # 一致性验证
                label_info = perform_enhanced_consistency_check(
                    field_data, layer_num, all_wavelengths, 
                    enhanced_evaluation_regions, enhanced_detector_labels
                )
                all_label_info.extend(label_info)
                
                # 生成一致性报告
                consistency_reports[layer_num] = generate_enhanced_consistency_report(
                    label_info, enhanced_evaluation_regions
                )
                
                print(f"    ✅ 第{layer_num}层: 可见度={visibility:.4f}")
            else:
                print(f"    ❌ 第{layer_num}层: 数据处理失败")
        
        # 4. 生成增强可视化
        print("\n🎨 生成增强版可视化...")
        vis_save_dir = os.path.join(save_dir, "enhanced_visualizations")
        
        for wavelength in all_wavelengths:
            wl_nm = int(wavelength * 1e9)
            for layer_num in all_layers:
                if layer_num in cross_matrices:
                    visualize_enhanced_detection_regions(
                        field_data, wavelength, layer_num, enhanced_evaluation_regions,
                        enhanced_detector_labels, region_info, all_label_info, vis_save_dir
                    )
        
        # 5. 绘制增强交叉矩阵图
        matrix_save_dir = os.path.join(save_dir, "enhanced_matrices")
        plot_enhanced_cross_matrices(
            normalized_matrices, visibility_results, all_wavelengths,
            all_layers, all_modes, enhanced_detector_labels, region_info,
            consistency_reports, matrix_save_dir
        )
        
        # 6. 保存增强数据
        data_save_dir = os.path.join(save_dir, "enhanced_data")
        save_enhanced_data(
            cross_matrices, normalized_matrices, visibility_results,
            all_wavelengths, all_layers, all_modes, enhanced_detector_labels,
            region_info, all_label_info, consistency_reports, data_save_dir
        )
        
        # 7. 打印增强摘要
        print_enhanced_summary(
            cross_matrices, normalized_matrices, visibility_results,
            all_wavelengths, all_modes, enhanced_detector_labels,
            region_info, all_label_info, consistency_reports
        )
        
        print(f"\n🎉 增强版分析完成！结果保存在: {save_dir}")
        
    except Exception as e:
        print(f"❌ 增强版分析过程中出现错误: {str(e)}")
        import traceback
        traceback.print_exc()
def generate_enhanced_region_info(evaluation_regions, detector_labels, wavelengths, modes):
    """生成增强的检测区域信息"""
    
    print("   🎯 生成增强检测区域映射...")
    
    region_info = []
    detector_idx = 0
    
    for wl_idx, wavelength in enumerate(wavelengths):
        wl_nm = int(wavelength * 1e9)
        
        for mode_idx in modes:
            if detector_idx < len(evaluation_regions):
                x_start, x_end, y_start, y_end = evaluation_regions[detector_idx]
                
                region_info.append({
                    'detector_idx': detector_idx,
                    'label': detector_labels.get(detector_idx, f'Det{detector_idx}'),
                    'wavelength_nm': wl_nm,
                    'mode_idx': mode_idx,
                    'center': ((x_start + x_end) / 2, (y_start + y_end) / 2),
                    'region': (x_start, x_end, y_start, y_end),
                    'size': (x_end - x_start, y_end - y_start)
                })
                
                detector_idx += 1
    
    print(f"   ✅ 生成 {len(region_info)} 个检测区域信息")
    return region_info


def calculate_enhanced_cross_matrix(field_data, layer_num, wavelengths, modes, 
                                  evaluation_regions, detector_labels):
    """计算增强版交叉矩阵"""
    
    print(f"    🔬 计算第{layer_num}层的增强交叉矩阵...")
    
    num_modes = len(modes)
    num_detectors = len(evaluation_regions)
    cross_matrix = np.zeros((num_detectors, num_modes))
    
    detector_idx = 0
    
    for wl_idx, wavelength in enumerate(wavelengths):
        wl_nm = int(wavelength * 1e9)
        print(f"    🔍 计算{wl_nm}nm模式的强度交叉矩阵...")
        
        if wavelength not in field_data or layer_num not in field_data[wavelength]:
            print(f"    ⚠️ 缺少{wl_nm}nm第{layer_num}层数据")
            detector_idx += num_modes
            continue
        
        layer_data = field_data[wavelength][layer_num]
        
        for mode_idx in modes:
            if detector_idx < num_detectors:
                x_start, x_end, y_start, y_end = evaluation_regions[detector_idx]
                
                # 计算该检测器对所有输入模式的响应
                for input_mode_idx, input_mode in enumerate(modes):
                    if input_mode in layer_data and layer_data[input_mode]:
                        intensity_data = layer_data[input_mode][0]['intensity']
                        
                        # 提取检测区域的强度
                        if (y_end <= intensity_data.shape[0] and x_end <= intensity_data.shape[1]):
                            region_intensity = intensity_data[y_start:y_end, x_start:x_end]
                            
                            # 使用加权平均计算区域强度
                            weighted_intensity = calculate_weighted_region_intensity(
                                region_intensity, (x_start, x_end, y_start, y_end)
                            )
                            
                            cross_matrix[detector_idx, input_mode_idx] = weighted_intensity
                        else:
                            print(f"    ⚠️ 检测器{detector_idx}区域超出数据范围")
                
                detector_idx += 1
    
    return cross_matrix

def load_enhanced_field_data(base_dir, wavelengths, layers, modes):
    """加载增强版场数据 - 模拟数据加载函数"""
    
    print("   📊 模拟加载增强版场数据...")
    
    field_data = {}
    
    for wavelength in wavelengths:
        wl_nm = int(wavelength * 1e9)
        field_data[wavelength] = {}
        
        for layer_num in layers:
            field_data[wavelength][layer_num] = {}
            
            for mode_idx in modes:
                # 模拟生成增强的场强度数据
                np.random.seed(42 + layer_num * 10 + mode_idx)  # 确保可重复性
                
                # 创建更真实的光场分布
                x = np.linspace(-5, 5, 500)
                y = np.linspace(-5, 5, 500)
                X, Y = np.meshgrid(x, y)
                
                # 根据波长和模式创建不同的场分布
                if wl_nm == 1310:
                    # 1310nm的场分布
                    if mode_idx == 1:
                        intensity = np.exp(-(X**2 + Y**2)/2) * (1 + 0.3*np.cos(2*np.pi*X))
                    elif mode_idx == 2:
                        intensity = np.exp(-((X-1)**2 + (Y-1)**2)/1.5) * (1 + 0.2*np.sin(np.pi*Y))
                    else:
                        intensity = np.exp(-((X+1)**2 + (Y+1)**2)/1.8) * (1 + 0.4*np.cos(np.pi*X*Y))
                else:
                    # 1550nm的场分布
                    if mode_idx == 1:
                        intensity = np.exp(-((X-0.5)**2 + (Y+0.5)**2)/2.2) * (1 + 0.25*np.sin(3*np.pi*X))
                    elif mode_idx == 2:
                        intensity = np.exp(-(X**2 + (Y-0.8)**2)/1.7) * (1 + 0.35*np.cos(2*np.pi*Y))
                    else:
                        intensity = np.exp(-((X+0.8)**2 + Y**2)/2.5) * (1 + 0.3*np.sin(np.pi*(X+Y)))
                
                # 添加层数相关的变化
                intensity *= (1 + 0.1 * layer_num)
                
                # 添加噪声
                noise = np.random.normal(0, 0.05, intensity.shape)
                intensity += noise
                intensity = np.maximum(intensity, 0)  # 确保非负
                
                field_data[wavelength][layer_num][mode_idx] = [{
                    'intensity': intensity,
                    'phase': np.random.random(intensity.shape) * 2 * np.pi,
                    'wavelength': wavelength,
                    'layer': layer_num,
                    'mode': mode_idx
                }]
    
    print(f"   ✅ 成功加载 {len(wavelengths)} 波长 × {len(layers)} 层 × {len(modes)} 模式的数据")
    return field_data

# 运行增强版分析
if __name__ == "__main__":
    main_enhanced_analysis()




多模式多波长光场调制系统 - 训练-仿真集成版 (完整修改版 - 增强一致性)
✓ 基准波长设置: 索引 1 -> 1550.0nm
✓ Zero Padding 配置: 填充比例=0.01, 衰减=True
  - 衰减类型: cosine, 宽度: 15像素
配置完成，使用设备: cuda
✅ 配置创建成功！
波长数量: 2
模式数量: 3
保存目录: ./results/2_wl_basewl_1.55e-06_z_prop_0.00015_focus_10/
🚀 启动增强版多波长强度交叉矩阵分析

📂 加载增强版数据...
   📊 模拟加载增强版场数据...
   ✅ 成功加载 2 波长 × 5 层 × 3 模式的数据

🎯 生成增强检测区域信息...
   🎯 生成增强检测区域映射...
   ✅ 生成 6 个检测区域信息

🔬 执行增强版强度交叉矩阵分析...

  分析第1层...
❌ 增强版分析过程中出现错误: name 'calculate_enhanced_cross_matrix' is not defined


Traceback (most recent call last):
  File "/tmp/ipykernel_34233/4263198111.py", line 1295, in main_enhanced_analysis
    cross_matrix = calculate_enhanced_cross_matrix(
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
NameError: name 'calculate_enhanced_cross_matrix' is not defined. Did you mean: 'plot_enhanced_cross_matrices'?
